In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
import optuna
import plotly
from warnings import filterwarnings
import matplotlib.pyplot as plt
filterwarnings("ignore")

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
data = load_breast_cancer()
X = data.data
y = data.target
feature_names = data.feature_names
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

def objective_svm(trial):
    C = trial.suggest_loguniform('C', 1e-2, 1e2)
    kernel = trial.suggest_categorical('kernel', ['linear', 'rbf', 'sigmoid'])
    gamma = trial.suggest_categorical('gamma', ['scale', 'auto'])

    model = SVC(C=C, kernel=kernel, gamma=gamma, random_state=42)

    scores = cross_validate(model, X_train, y_train, cv=3, scoring='accuracy')
    return scores['test_score'].mean()

study_svm = optuna.create_study(direction='maximize')
study_svm.optimize(objective_svm, n_trials=50)

[I 2026-05-24 16:49:31,129] A new study created in memory with name: no-name-da715466-999d-48be-8992-cf67b78c65de


[I 2026-05-24 16:49:31,144] Trial 0 finished with value: 0.6285726734053677 and parameters: {'C': 0.07127394731288197, 'kernel': 'sigmoid', 'gamma': 'scale'}. Best is trial 0 with value: 0.6285726734053677.
[I 2026-05-24 16:49:31,158] Trial 1 finished with value: 0.6285726734053677 and parameters: {'C': 0.40544562131894823, 'kernel': 'sigmoid', 'gamma': 'scale'}. Best is trial 0 with value: 0.6285726734053677.
[I 2026-05-24 16:49:35,130] Trial 2 finished with value: 0.953831183920065 and parameters: {'C': 16.76605497924637, 'kernel': 'linear', 'gamma': 'auto'}. Best is trial 2 with value: 0.953831183920065.
[I 2026-05-24 16:49:35,151] Trial 3 finished with value: 0.6285726734053677 and parameters: {'C': 0.436209693500296, 'kernel': 'rbf', 'gamma': 'auto'}. Best is trial 2 with value: 0.953831183920065.
[I 2026-05-24 16:49:35,160] Trial 4 finished with value: 0.9208202625769722 and parameters: {'C': 36.66133709028815, 'kernel': 'rbf', 'gamma': 'scale'}. Best is trial 2 with value: 0.953

In [3]:
print("Najlepsze hiperparametry dla SVM:")
print(study_svm.best_params)
print("Najlepsza dokładność:", study_svm.best_value)

Najlepsze hiperparametry dla SVM:
{'C': 8.658874381141596, 'kernel': 'linear', 'gamma': 'scale'}
Najlepsza dokładność: 0.9626321598698734


In [4]:
optuna.visualization.plot_optimization_history(study_svm)

In [5]:
optuna.visualization.plot_param_importances(study_svm)

In [6]:
optuna.visualization.plot_contour(study_svm, params=['kernel','gamma'])

In [7]:
best_svm = SVC(**study_svm.best_params, random_state=42)
best_svm.fit(X_train, y_train)
y_pred_svm = best_svm.predict(X_test)
print("Dokładność na zbiorze testowym (SVM):", accuracy_score(y_test, y_pred_svm))

Dokładność na zbiorze testowym (SVM): 0.956140350877193


In [ ]:
def objective_rf(trial):
    n_estimators = trial.suggest_int('n_estimators', 10, 200)
    max_depth = trial.suggest_int('max_depth', 2, 20)
    min_samples_split = trial.suggest_int('min_samples_split',2, 20)
    min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 20)

    model = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth,
                                   min_samples_split=min_samples_split,
                                   min_samples_leaf=min_samples_leaf,
                                   random_state=42)

    scores = cross_validate(model, X_train, y_train, cv=3, scoring='accuracy')
    return scores['test_score'].mean()
study_rf_10 = optuna.create_study(direction='maximize')
study_rf_10.optimize(objective_rf, n_trials=10)

study_rf_100 = optuna.create_study(direction='maximize')
study_rf_100.optimize(objective_rf, n_trials=100)

print("10 prób:", study_rf_10.best_value, study_rf_10.best_params)
print("100 prób:", study_rf_100.best_value, study_rf_100.best_params)

[I 2026-05-24 17:10:21,261] A new study created in memory with name: no-name-56f51878-97b1-4c00-97d1-aee9afee9aa8
[I 2026-05-24 17:10:21,544] Trial 0 finished with value: 0.9384657836644591 and parameters: {'n_estimators': 83, 'max_depth': 19, 'min_samples_split': 12, 'min_samples_leaf': 8}. Best is trial 0 with value: 0.9384657836644591.
[I 2026-05-24 17:10:21,997] Trial 1 finished with value: 0.9340362495643081 and parameters: {'n_estimators': 158, 'max_depth': 20, 'min_samples_split': 7, 'min_samples_leaf': 15}. Best is trial 0 with value: 0.9384657836644591.
[I 2026-05-24 17:10:22,461] Trial 2 finished with value: 0.9384367375392122 and parameters: {'n_estimators': 137, 'max_depth': 19, 'min_samples_split': 20, 'min_samples_leaf': 13}. Best is trial 0 with value: 0.9384657836644591.
[I 2026-05-24 17:10:22,770] Trial 3 finished with value: 0.9472377134890205 and parameters: {'n_estimators': 109, 'max_depth': 16, 'min_samples_split': 15, 'min_samples_leaf': 5}. Best is trial 3 with v

10 prób: 0.9472377134890205 {'n_estimators': 109, 'max_depth': 16, 'min_samples_split': 15, 'min_samples_leaf': 5}
100 prób: 0.9603956082258627 {'n_estimators': 118, 'max_depth': 9, 'min_samples_split': 3, 'min_samples_leaf': 1}


In [13]:
optuna.visualization.plot_optimization_history(study_rf_10)

In [14]:
optuna.visualization.plot_optimization_history(study_rf_100)

In [19]:
optuna.visualization.plot_param_importances(study_rf_10)

In [20]:
optuna.visualization.plot_param_importances(study_rf_100)

In [24]:
optuna.visualization.plot_contour(study_rf_10, params=['min_samples_leaf' ,'n_estimators'])

In [26]:
best_rf_100 = RandomForestClassifier(**study_rf_100.best_params, random_state=42)
best_rf_100.fit(X_train, y_train)
y_pred_rf_100 = best_rf_100.predict(X_test)
print("Dokładność na zbiorze testowym (Random Forest):", accuracy_score(y_test, y_pred_rf_100))

optuna.visualization.plot_parallel_coordinate(study_rf_100, params=['n_estimators', 'max_depth', 'min_samples_split', 'min_samples_leaf'])

Dokładność na zbiorze testowym (Random Forest): 0.9649122807017544


In [30]:
def multi_objective_rf(trial):
    n_estimators = trial.suggest_int('n_estimators', 10, 200)
    max_depth = trial.suggest_int('max_depth', 2, 20)
    min_samples_split = trial.suggest_int('min_samples_split',2, 20)
    min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 20)

    model = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth,
                                   min_samples_split=min_samples_split,
                                   min_samples_leaf=min_samples_leaf,
                                   random_state=42)

    scores = cross_validate(model, X_train, y_train, cv=3, scoring=['accuracy', 'f1'])
    acc = scores['test_accuracy'].mean()
    f1 = scores['test_f1'].mean()
    return acc, f1

study_multi = optuna.create_study(directions=['minimize', 'minimize'])
study_multi.optimize(multi_objective_rf, n_trials=50)



[I 2026-05-26 17:49:09,234] A new study created in memory with name: no-name-d0a23d06-bf0e-4cc7-baca-ae1139fad265
[I 2026-05-26 17:49:09,508] Trial 0 finished with values: [0.934065295689555, 0.9478899197413142] and parameters: {'n_estimators': 86, 'max_depth': 3, 'min_samples_split': 14, 'min_samples_leaf': 5}.
[I 2026-05-26 17:49:09,547] Trial 1 finished with values: [0.9274137330080167, 0.943191893206823] and parameters: {'n_estimators': 10, 'max_depth': 17, 'min_samples_split': 19, 'min_samples_leaf': 13}.
[I 2026-05-26 17:49:09,710] Trial 2 finished with values: [0.9318432671081678, 0.9461681579848434] and parameters: {'n_estimators': 56, 'max_depth': 3, 'min_samples_split': 4, 'min_samples_leaf': 13}.
[I 2026-05-26 17:49:10,250] Trial 3 finished with values: [0.922998721970489, 0.939997698976927] and parameters: {'n_estimators': 199, 'max_depth': 11, 'min_samples_split': 16, 'min_samples_leaf': 18}.
[I 2026-05-26 17:49:10,351] Trial 4 finished with values: [0.9428227024514929, 0.

In [31]:
optuna.visualization.plot_pareto_front(study_multi, target_names=['Accuracy', 'F1 Score'])